In [1]:
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

buoy = 'WLIS'

# Load dataset
df = pd.read_csv(f'{buoy}_sfc_QAQC.csv', usecols=['TmStamp','T_data','T_Q'])
df['TmStamp'] = pd.to_datetime(df['TmStamp'], format='%d-%b-%Y %H:%M:%S', errors='coerce')
df = df[df['TmStamp'] < '2026-01-01']
df = df.sort_values('TmStamp').set_index('TmStamp')

# Remove failed QA/QC records
df['T_Q'] = df['T_Q'].astype(str).str.zfill(5)
df = df[df['T_Q'].str[0] != '4'][['T_data']]

# Resampling and interpolation
df = df.resample('15min').mean()
df['T_data'] = df['T_data'].interpolate(limit=8)
df = df.dropna()

# 40-hour low-pass filter (remove tidal + diurnal)
# Sampling frequency (96 samples/day)
fs, cutoff_hours = 96, 40
b, a = butter(N=4, Wn=(1/cutoff_hours) / (fs/2), btype='low')
df['SST'] = filtfilt(b, a, df['T_data'])

# Daily mean SST and gap interpolation
daily_SST = df['SST'].resample('D').mean()
daily_SST = daily_SST.interpolate(limit=1)

# DOY climatology (mean & 90th percentile)
df_T = pd.DataFrame({'SST': daily_SST}).dropna()
df_T = df_T[~((df_T.index.month == 2) & (df_T.index.day == 29))]
df_T['DOY'] = df_T.index.dayofyear
clim_mean = df_T.groupby('DOY')['SST'].mean()
clim_p90 = df_T.groupby('DOY')['SST'].quantile(0.9)

# 11-day moving window
clim_mean = clim_mean.rolling(11, center=True, min_periods=1).mean()
clim_p90  = clim_p90.rolling(11, center=True, min_periods=1).mean()
df_T['clim_mean'] = df_T['DOY'].map(clim_mean)
df_T['clim_p90'] = df_T['DOY'].map(clim_p90)

# MHW identification
mhw = df_T['SST'] > df_T['clim_p90']
grp = mhw.ne(mhw.shift()).cumsum()

# Fill in gap ≤ 2 days
gap_sizes = (~mhw).groupby(grp).transform('sum')
mhw_filled = mhw | (gap_sizes <= 2)
grp_filled = mhw_filled.ne(mhw_filled.shift()).cumsum()

# Extract MHW events (duration ≥ 5 days)
events = df_T[mhw_filled].groupby(grp_filled)
mhw_events = pd.DataFrame(
    [(g.index[0], g.index[-1]) for _, g in events if len(g) >= 5],
    columns=['start', 'end'])

# MHW events
df_T['year'] = df_T.index.year
df_T['intensity'] = df_T['SST'] - df_T['clim_mean']
df_T['event_id'] = np.nan
for i, (start, end) in mhw_events.iterrows():
    df_T.loc[start:end, 'event_id'] = i
df_T['is_mhw'] = df_T['event_id'].notna()

# Processed dataset output
df_T_out = df_T[['SST','clim_mean','clim_p90','is_mhw']]
# df_T_out.reset_index().rename(columns={'TmStamp':'date'}).to_csv(f'{buoy}_SST.csv', index=False)
df_T_out.head()

,SST,clim_mean,clim_p90,is_mhw
TmStamp,,,,
2000-09-06,22.304952,22.947769,24.128342,False
2000-09-07,22.298700,22.885550,24.077677,False
2000-09-08,22.283813,22.821164,24.016163,False
2000-09-09,22.261740,22.754598,23.941281,False
2000-09-10,22.231903,22.685765,23.855001,False


In [2]:
# MHW metrics
mhw_metrics = (
    df_T.dropna(subset=['event_id']).groupby('event_id')
        .agg(start=('SST', lambda x: x.index[0]),
             end=('SST', lambda x: x.index[-1]),
             duration=('SST', 'size'),
             mean_intensity=('intensity', 'mean'),
             cum_intensity=('intensity', 'sum'))
        .round(3).reset_index(drop=True)
)

# mhw_metrics.to_csv(f'{buoy}_metrics.csv', index=False)
mhw_metrics

,start,end,duration,mean_intensity,cum_intensity
0,2001-06-16,2001-06-24,9,1.263,11.368
1,2001-11-30,2001-12-28,29,2.525,73.213
2,2002-01-26,2002-03-07,17,3.373,57.347
3,2002-08-07,2002-08-16,10,0.896,8.959
4,2003-03-15,2003-03-19,5,2.719,13.595
5,2003-12-26,2004-01-02,8,4.395,35.160
6,2004-10-16,2004-11-04,20,2.026,40.522
7,2005-07-22,2005-08-12,21,1.330,27.934
8,2005-09-18,2005-10-07,20,1.128,22.568
9,2006-02-23,2006-02-27,5,4.594,22.972


In [3]:
# Annual MHW metrics
annual_metrics = pd.DataFrame({
    'frequency': mhw_metrics['start'].dt.year.value_counts(),
    'MHW_days': df_T[df_T['is_mhw']].groupby('year').size(),
    'annual_cum_intensity': df_T[df_T['is_mhw']].groupby('year')['intensity'].sum().round(3)
}).fillna(0).sort_index().rename_axis('year').reset_index()

# annual_metrics.to_csv(f'{buoy}_annual_metrics.csv', index=False)
annual_metrics

,year,frequency,MHW_days,annual_cum_intensity
0,2001,2.0,38,84.582
1,2002,2.0,27,66.306
2,2003,2.0,11,36.398
3,2004,1.0,22,52.880
4,2005,2.0,41,50.502
5,2006,1.0,5,22.972
6,2007,2.0,52,99.228
7,2008,1.0,7,6.761
8,2010,3.0,48,84.951
9,2011,1.0,18,39.722
